# 01. Hyperparameter selection by spatial cross-validation

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.transfer import transfer
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

## Search space

In [ ]:
n_estimators = 300
search_space = {"max_features": ["sqrt", .2, .4], "min_samples_leaf": [1, 2, 4, 8, 16],
                "max_samples": [.5, .8], "max_depth": [None, 12]}
grid = tuning.parameter_grid(search_space, n_estimators=n_estimators)
print("Settings per cohort:", len(grid))

## Run spatial cross-validation

In [ ]:
run = tuning.new_run(root, "spatial_selection", features_path, area=transfer.AREA, n_estimators=n_estimators,
                     search_space={k: [str(v) for v in values] for k, values in search_space.items()})
records = []
for cohort in cohorts:
    train, _ = tuning.cohort_split(development, cohort)
    for number, params in enumerate(grid, 1):
        pooled = transfer.spatial_cv(train, params)
        records.append({"cohort": cohort, **params, "spatial_macro_f1": pooled["macro_f1"],
                        "spatial_balanced_accuracy": pooled["balanced_accuracy"],
                        "fold_min": min(pooled["fold_macro_f1"]), "fold_max": max(pooled["fold_macro_f1"]),
                        **{f"{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()}})
        if number % 10 == 0 or number == len(grid):
            print(cohort, f"{number}/{len(grid)}", flush=True)
print("Saved:", tuning.save_records(run, "spatial_grid", records))

## Results table

In [ ]:
results = frame(records)
for cohort in cohorts:
    print(cohort)
    display(results[results.cohort == cohort].sort_values("spatial_macro_f1", ascending=False).head(10).round(3))

## Selected settings

In [ ]:
selected = {}
for cohort in cohorts:
    best = max((r for r in records if r["cohort"] == cohort), key=lambda r: (r["spatial_macro_f1"], r["fold_min"]))
    selected[cohort] = {**{k: best[k] for k in tuning.PARAMETERS}, "n_estimators": 1000}
(run / "selected.json").write_text(json.dumps(selected, indent=2))
selected

## Selected settings compared with current settings

In [ ]:
check = []
for cohort in cohorts:
    train, validation = tuning.cohort_split(development, cohort)
    for role, params in (("current_leaf_2", tuning.BASELINE), ("current_leaf_4", {**tuning.BASELINE, "min_samples_leaf": 4}),
                         ("spatial_selected", selected[cohort])):
        pooled = transfer.spatial_cv(train, params)
        locked = transfer.locked_validation(train, validation, params)
        check.append({"cohort": cohort, "role": role, **params, "spatial_macro_f1": pooled["macro_f1"],
                      "locked_validation_macro_f1": locked["macro_f1"]})
tuning.save_records(run, "selected_vs_current", check)
display(frame(check).round(3))

## Parameter effects

In [ ]:
parameters = list(search_space)
figure, axes = plt.subplots(1, len(parameters), figsize=(18, 4), sharey=True)
for axis, parameter in zip(axes, parameters):
    results.groupby([results[parameter].astype(str), "cohort"]).spatial_macro_f1.mean().unstack().plot(
        kind="bar", ax=axis, legend=axis is axes[-1])
    axis.set_title(parameter)
    axis.set_ylabel("Mean spatial CV parcel macro-F1")
show_and_save(figure, run, "parameter_effects")
print("Run folder:", run)